# 09 · Sync Outputs ↔ Google Drive

Notebook para sincronização manual de outputs, workflows e logs entre o Kaggle Notebook e o Google Drive.

**Uso:**
- Execute após gerar imagens no ComfyUI para subir ao Drive (`push`)
- Execute antes de iniciar para baixar workflows compartilhados (`pull`)

**Pré-requisitos no Kaggle Secrets:**
- `GDRIVE_SERVICE_ACCOUNT_JSON`: Service account JSON com acesso ao Drive

In [ ]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("/kaggle/working/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))

from kaggle_drive_sync import sync_outputs, get_drive_path, detect_env

# Configurações
DRIVE_BASE = "Automa/ComfyUI"
LOCAL_OUTPUTS = Path("/kaggle/working/ComfyUI/output")
LOCAL_WORKFLOWS = Path("/kaggle/working/ComfyUI/user")

print(f"Ambiente detectado: {detect_env()}")
print(f"Drive base: {DRIVE_BASE}")
print(f"Local outputs: {LOCAL_OUTPUTS}")
print(f"Local workflows: {LOCAL_WORKFLOWS}")

In [ ]:
# Testar conexão com Drive
try:
    drive_path = get_drive_path(drive_base=DRIVE_BASE, env="kaggle")
    print(f"✅ Drive acessível: {drive_path}")
    for subdir in ["outputs", "workflows", "logs", "metadata"]:
        d = drive_path / subdir
        d.mkdir(parents=True, exist_ok=True)
        print(f"  📁 {subdir}: {d}")
except Exception as e:
    print(f"❌ Falha ao acessar Drive: {e}")
    raise

## Push: Enviar outputs locais → Google Drive

Use após gerar imagens no ComfyUI para fazer backup no Drive.

In [ ]:
# PUSH: Local → Drive
stats = sync_outputs(
    action="push",
    local_outputs=LOCAL_OUTPUTS,
    local_workflows=LOCAL_WORKFLOWS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)

print(f"\n✅ Push concluído")
print(f"  Enviados: {stats['synced']}")
print(f"  Pulados (idênticos): {stats['skipped']}")
print(f"  Erros: {stats['errors']}")

## Pull: Baixar do Google Drive → Local

Use para baixar workflows compartilhados ou restaurar backup.

In [ ]:
# PULL: Drive → Local (apenas workflows por padrão)
stats = sync_outputs(
    action="pull",
    local_outputs=LOCAL_OUTPUTS,
    local_workflows=LOCAL_WORKFLOWS,
    drive_base=DRIVE_BASE,
    env="kaggle",
)

print(f"\n✅ Pull concluído")
print(f"  Baixados: {stats['synced']}")
print(f"  Pulados (idênticos): {stats['skipped']}")
print(f"  Erros: {stats['errors']}")

## Verificar estrutura no Drive

Lista o que está salvo no Google Drive.

In [ ]:
# Listar arquivos no Drive
drive_path = get_drive_path(drive_base=DRIVE_BASE, env="kaggle")

for subdir in ["outputs", "workflows", "logs", "metadata"]:
    d = drive_path / subdir
    if d.exists():
        files = list(d.rglob("*"))
        files = [f for f in files if f.is_file()]
        total_mb = sum(f.stat().st_size for f in files) / (1024**2)
        print(f"{subdir}: {len(files)} arquivos, {total_mb:.1f} MB")
        for f in sorted(files)[:10]:
            size_mb = f.stat().st_size / (1024**2)
            rel = f.relative_to(d)
            print(f"  {rel} ({size_mb:.1f} MB)")
        if len(files) > 10:
            print(f"  ... e mais {len(files) - 10} arquivo(s)")
    else:
        print(f"{subdir}: (vazio)")